In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-24T15:23:28.475787+00:00', 'open': 98.44, 'high': 101.53, 'low': 97.15, 'close': 100.04, 'volume': 502, 'trade_count': 30, 'vwap': 100.35}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-24T15:23:28.473771+00:00', 'open': 102.56, 'high': 105.46, 'low': 103.26, 'close': 104.3, 'volume': 180, 'trade_count': 32, 'vwap': 104.37}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-24T15:23:28.467770+00:00', 'open': 94.4, 'high': 96.85, 'low': 93.91, 'close': 95.32, 'volume': 136, 'trade_count': 45, 'vwap': 96.06}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-24T15:23:29.525604+00:00', 'open': 98.46, 'high': 101.28, 'low': 98.02, 'close': 100.33, 'volume': 104, 'trade_count': 49, 'vwap': 99.96}
Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-24T15:23:29.525604+00:00', 'open': 102.77, 'high': 105.42, 'low': 103.22, 'close': 104.42, 'volume': 104, 'trade_count': 40, 'vwap': 105.39}
Pushed to te